In [1]:
import data_helper
df = data_helper.load('h',0,1,1,1)

In [2]:
import pandas as pd

df['load_lag_48h'] = df['load'].shift(48)
df['load_lag_168h'] = df['load'].shift(168)
df['load_lag_168h'] = df['load'].shift(336)
df = df.dropna()

In [4]:
df['Total_Wind'] = df['Wind Offshore'] +  df['Wind Onshore'] 

In [ ]:
import numpy as np

df['Wind_Log'] = np.log1p(df['Total_Wind'])

noise_threshold = 0.1 

# If physical solar generation exceeds the noise floor, keep it. 
# Otherwise, force it to absolute zero.
df['Solar_Gated'] = np.where(df['Solar'] > noise_threshold, df['Solar'], 0.0)
df['Solar_Log'] = np.log1p(df['Solar_Gated'])

In [ ]:
# School holiday ranges
school_breaks = [
    # 2024
    ('2024-03-23', '2024-04-01'), # Easter 2024
    ('2024-05-09', '2024-05-12'), # Ascension Day 2024
    ('2024-05-18', '2024-05-20'), # Whitsun 2024
    ('2024-06-29', '2024-08-11'), # Summer 2024
    ('2024-10-12', '2024-10-20'), # Autumn 2024
    ('2024-12-21', '2025-01-05'), # Christmas 2024
    
    # 2025
    ('2025-02-08', '2025-02-16'), # Winter 2025
    ('2025-04-12', '2025-04-21'), # Easter 2025
    ('2025-05-29', '2025-06-01'), # Ascension Day 2025
    ('2025-06-07', '2025-06-09'), # Whitsun 2025
    ('2025-06-28', '2025-08-10'), # Summer 2025
    ('2025-10-11', '2025-10-19'), # Autumn 2025
    ('2025-12-23', '2026-01-04'), # Christmas 2025
    
    # 2026
    ('2026-02-07', '2026-02-15'), # Winter 2026
    ('2026-03-28', '2026-04-06')  # Easter 2026
]

# Initialize the flag
df['is_school_break'] = 0

# Apply the flags
tz = df.index.tz
for start, end in school_breaks:
    start_ts = pd.Timestamp(start, tz=tz)
    end_ts = pd.Timestamp(end, tz=tz) + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
    
    df.loc[start_ts:end_ts, 'is_school_break'] = 1

df['is_prev_week_school_break'] = df['is_school_break'].shift(168).fillna(0).astype(int)

In [8]:
df_full = df.copy()

In [ ]:
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_percentage_error

# ==========================================
# FEATURE SPACE
# ==========================================

FEATURES = [
    'hour', 'dayofweek', 'month', 
    'Total_Wind', 'Solar_Log',
    'DT1', 'DT2', 'DT3', 
    'TL2W_w', 'HDD',
    'load_lag_48h', 'load_lag_168h', 'is_school_break', 'is_prev_week_school_break'
]
TARGET = 'load'

# localized to UTC
if 'timestamp' in df_full.columns:
    df_full = df_full.set_index('timestamp')

df_xgb = df_full.copy()

# ==========================================
# CATEGORICAL TIMESTAMPS
# ==========================================

df_xgb['hour'] = df_xgb.index.hour
df_xgb['dayofweek'] = df_xgb.index.dayofweek
df_xgb['month'] = df_xgb.index.month


df_xgb = df_xgb.dropna(subset=FEATURES + [TARGET])

# ==========================================
# GLOBAL XGBOOST
# ==========================================
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=1500,
    learning_rate=0.01,
    max_depth=5,
    subsample=0.7,
    colsample_bytree=0.6,
    tree_method='hist',
    early_stopping_rounds=50,
    random_state=42
)

# ==========================================
# TS-CV
# ==========================================
end_date = pd.to_datetime('2026-03-01 00:00:00', utc=True)
test_start_dates = [end_date - pd.Timedelta(days=i) for i in range(56, 0, -1)]

xgb_forecasts = []
fold_mapes = []

for i, test_start in enumerate(test_start_dates, 1):
    # horizon = 36 hours
    test_end = test_start + pd.Timedelta(hours=35)
    
    # SPLIT DATA
    train_mask = df_xgb.index < test_start
    test_mask = (df_xgb.index >= test_start) & (df_xgb.index <= test_end)
    
    val_start = test_start - pd.Timedelta(days=7)
    val_mask = (df_xgb.index >= val_start) & (df_xgb.index < test_start)
    pure_train_mask = train_mask & ~val_mask

    X_train, y_train = df_xgb.loc[pure_train_mask, FEATURES], df_xgb.loc[pure_train_mask, TARGET]
    X_val, y_val = df_xgb.loc[val_mask, FEATURES], df_xgb.loc[val_mask, TARGET]
    X_test, y_test = df_xgb.loc[test_mask, FEATURES], df_xgb.loc[test_mask, TARGET]
    
    # TRAIN
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False 
    )
    
    # PREDICT
    preds = xgb_model.predict(X_test)
    
    # STORE RESULTS
    df_pred = pd.DataFrame({
        'timestamp': X_test.index,
        'Actual': y_test.values,
        'XGB_Prediction': preds,
        'Fold': i
    })
    xgb_forecasts.append(df_pred)
    
    fold_mape = mean_absolute_percentage_error(y_test, preds)
    fold_mapes.append(fold_mape)
    
    #log for debug/fine-tune
    print(f"Fold {i}/56 completed. Horizon: {test_start.strftime('%Y-%m-%d')} | MAPE: {fold_mape*100:.2f}%")

# ==========================================
# FINAL RESULT
# ==========================================
df_xgb_results = pd.concat(xgb_forecasts, ignore_index=True)
global_xgb_mape = mean_absolute_percentage_error(df_xgb_results['Actual'], df_xgb_results['XGB_Prediction'])

print(f"\n==========================================")
print(f"XGBOOST (CATEGORICAL) BASELINE RESULTS")
print(f"Global Aggregated MAPE: {global_xgb_mape * 100:.2f}%")
print(f"==========================================")

Initiating 56-Fold Evaluation with Categorical Time Features...
Fold 1/56 completed. Horizon: 2026-01-04 | MAPE: 10.19%
Fold 2/56 completed. Horizon: 2026-01-05 | MAPE: 11.44%
Fold 3/56 completed. Horizon: 2026-01-06 | MAPE: 9.92%
Fold 4/56 completed. Horizon: 2026-01-07 | MAPE: 6.07%
Fold 5/56 completed. Horizon: 2026-01-08 | MAPE: 4.80%
Fold 6/56 completed. Horizon: 2026-01-09 | MAPE: 3.79%
Fold 7/56 completed. Horizon: 2026-01-10 | MAPE: 8.26%
Fold 8/56 completed. Horizon: 2026-01-11 | MAPE: 8.79%
Fold 9/56 completed. Horizon: 2026-01-12 | MAPE: 4.63%
Fold 10/56 completed. Horizon: 2026-01-13 | MAPE: 5.84%
Fold 11/56 completed. Horizon: 2026-01-14 | MAPE: 5.83%
Fold 12/56 completed. Horizon: 2026-01-15 | MAPE: 3.47%
Fold 13/56 completed. Horizon: 2026-01-16 | MAPE: 3.52%
Fold 14/56 completed. Horizon: 2026-01-17 | MAPE: 3.68%
Fold 15/56 completed. Horizon: 2026-01-18 | MAPE: 3.03%
Fold 16/56 completed. Horizon: 2026-01-19 | MAPE: 3.22%
Fold 17/56 completed. Horizon: 2026-01-20 | MAP

In [ ]:
from  sklearn.metrics import root_mean_squared_error
from  sklearn.metrics import mean_absolute_error
# ==========================================
# FINAL RESULT
# ==========================================
df_xgb_results = pd.concat(xgb_forecasts, ignore_index=True)
global_xgb_mape = mean_absolute_percentage_error(df_xgb_results['Actual'], df_xgb_results['XGB_Prediction'])
global_xgb_rmse = root_mean_squared_error(df_xgb_results['Actual'], df_xgb_results['XGB_Prediction'])
global_xgb_mae = mean_absolute_error(df_xgb_results['Actual'], df_xgb_results['XGB_Prediction'])

print(f"\n==========================================")
print(f"XGBOOST (CATEGORICAL) BASELINE RESULTS")
print(f"Total 36h Horizons Tested: 56")
print(f"MAPE: {global_xgb_mape * 100:.2f}%")
print(f"RMSE: {global_xgb_rmse :.2f} MW")
print(f"MAE: {global_xgb_mae:.2f} MW")
print(f"==========================================")


XGBOOST (CATEGORICAL) BASELINE RESULTS
Total 36h Horizons Tested: 56
MAPE: 5.45%
RMSE: 246.33 MW
MAE: 180.60 MW


In [ ]:
#Export results
file_name = 'xgboost_baseline_results.csv'
df_xgb_results.to_csv(file_name, index=False)

print(f"Results successfully saved to '{file_name}'")

Results successfully saved to 'xgboost_baseline_results.csv'
Shape of saved matrix: (2016, 4)


In [ ]:
import time

# ==========================================
# HOUR-SPECIFIC FEATURE SPACE
# ==========================================
# 'hour' is strictly removed. 'dayofweek' and 'month' remain to capture macro-seasonality.
FEATURES = [
    'dayofweek', 'month', 'is_holiday', 'Total_Wind', 
    'Solar_Log',
    'DT1', 'DT2', 'DT3', 'TL2W_w', 'HDD', 
    'load_lag_48h', 'load_lag_168h',
    'is_school_break', 'is_prev_week_school_break'
]
TARGET = 'load'

# timestamp localized
if 'timestamp' in df_full.columns:
    df_full = df_full.set_index('timestamp')

df_xgb = df_full.copy()
df_xgb['hour'] = df_xgb.index.hour
df_xgb['dayofweek'] = df_xgb.index.dayofweek
df_xgb['month'] = df_xgb.index.month

# Drop NaNs
df_xgb = df_xgb.dropna(subset=FEATURES + [TARGET, 'hour'])

# ==========================================
# TS-CV
# ==========================================
end_date = pd.to_datetime('2026-03-01 00:00:00', utc=True)
test_start_dates = [end_date - pd.Timedelta(days=i) for i in range(56, 0, -1)]

xgb_forecasts = []
fold_mapes = []

print("Initiating 56-Fold 24-Expert Walk-Forward Evaluation...")
start_time = time.time()

for i, test_start in enumerate(test_start_dates, 1):
    test_end = test_start + pd.Timedelta(hours=35)
    
    # Boundaries for the fold
    train_mask = df_xgb.index < test_start
    test_mask = (df_xgb.index >= test_start) & (df_xgb.index <= test_end)
    
    val_start = test_start - pd.Timedelta(days=7)
    val_mask = (df_xgb.index >= val_start) & (df_xgb.index < test_start)
    pure_train_mask = train_mask & ~val_mask

    fold_predictions = []

    # ==========================================
    # TRAIN & INFFER PER HOUR (24 EXPERTS)
    # ==========================================
    for h in range(24):
        # data specific to hour 'h'
        hour_mask = df_xgb['hour'] == h
        
        # Splitting
        X_train_h = df_xgb.loc[pure_train_mask & hour_mask, FEATURES]
        y_train_h = df_xgb.loc[pure_train_mask & hour_mask, TARGET]
        
        X_val_h = df_xgb.loc[val_mask & hour_mask, FEATURES]
        y_val_h = df_xgb.loc[val_mask & hour_mask, TARGET]
        
        X_test_h = df_xgb.loc[test_mask & hour_mask, FEATURES]
        y_test_h = df_xgb.loc[test_mask & hour_mask, TARGET]
        
        # safeguard
        if X_test_h.empty:
            continue
            
        # Expert init
        expert = xgb.XGBRegressor(
            objective='reg:squarederror',
            n_estimators=350,
            learning_rate=0.06,
            max_depth=3,
            subsample=0.66,
            colsample_bytree=0.6,
            tree_method='hist',
            early_stopping_rounds=25,
            random_state=42
        )
        
        # Training
        expert.fit(
            X_train_h, y_train_h,
            eval_set=[(X_val_h, y_val_h)],
            verbose=False
        )
        
        # Inference
        preds_h = expert.predict(X_test_h)
        
        # Store
        df_pred_h = pd.DataFrame({
            'timestamp': X_test_h.index,
            'Actual': y_test_h.values,
            'XGB_Prediction': preds_h,
            'Fold': i
        })
        fold_predictions.append(df_pred_h)

    # ==========================================
    # HORIZON RECONSTRUCTION
    # ==========================================
    # Concatenate the hourly predictions and sort strictly by timestamp 
    # to rebuild the contiguous 36-hour sequence.
    df_fold_reconstructed = pd.concat(fold_predictions).sort_values('timestamp')
    xgb_forecasts.append(df_fold_reconstructed)
    
    fold_mape = mean_absolute_percentage_error(
        df_fold_reconstructed['Actual'], 
        df_fold_reconstructed['XGB_Prediction']
    )
    fold_mapes.append(fold_mape)
    
    if i % 10 == 0 or i == 56:
        print(f"Fold {i}/56 completed. Horizon: {test_start.strftime('%Y-%m-%d')} | MAPE: {fold_mape*100:.2f}%")

# ==========================================
# FINAL RESULTS
# ==========================================
df_expert_results = pd.concat(xgb_forecasts, ignore_index=True)
expert_xgb_mape = mean_absolute_percentage_error(df_expert_results['Actual'], df_expert_results['XGB_Prediction'])
expert_xgb_rmse = 
total_time = (time.time() - start_time) / 60

print(f"\n==========================================")
print(f"XGBOOST (24-EXPERT) BASELINE RESULTS")
print(f"Total 36h Horizons Tested: 56")
print(f"Execution Time: {total_time:.1f} minutes")
print(f"Global Aggregated MAPE: {expert_xgb_mape * 100:.2f}%")
print(f"==========================================")

df_expert_results.to_csv('xgboost_24expert_results.csv', index=False)

Initiating 56-Fold 24-Expert Walk-Forward Evaluation...
Fold 10/56 completed. Horizon: 2026-01-13 | MAPE: 8.16%
Fold 20/56 completed. Horizon: 2026-01-23 | MAPE: 3.35%
Fold 30/56 completed. Horizon: 2026-02-02 | MAPE: 4.66%
Fold 40/56 completed. Horizon: 2026-02-12 | MAPE: 2.60%
Fold 50/56 completed. Horizon: 2026-02-22 | MAPE: 4.90%
Fold 56/56 completed. Horizon: 2026-02-28 | MAPE: 4.02%

XGBOOST (24-EXPERT) BASELINE RESULTS
Total 36h Horizons Tested: 56
Execution Time: 2.6 minutes
Global Aggregated MAPE: 6.40%


In [ ]:
# ==========================================
# FINAL RESULTS
# ==========================================
df_expert_results = pd.concat(xgb_forecasts, ignore_index=True)
expert_xgb_mape = mean_absolute_percentage_error(df_expert_results['Actual'], df_expert_results['XGB_Prediction'])
expert_xgb_rmse = root_mean_squared_error(df_expert_results['Actual'], df_expert_results['XGB_Prediction'])
expert_xgb_mae = mean_absolute_error(df_expert_results['Actual'], df_expert_results['XGB_Prediction'])
total_time = (time.time() - start_time) / 60

print(f"\n==========================================")
print(f"XGBOOST (24-EXPERT) BASELINE RESULTS")
print(f"Total 36h Horizons Tested: 56")
print(f"Execution Time: {total_time:.1f} minutes")
print(f"MAPE: {expert_xgb_mape * 100:.2f}%")
print(f"RSME: {expert_xgb_rmse:.2f} MW")
print(f"MAE: {expert_xgb_mae:.2f} MW")
print(f"==========================================")


XGBOOST (24-EXPERT) BASELINE RESULTS
Total 36h Horizons Tested: 56
Execution Time: 8.6 minutes
MAPE: 6.40%
RSME: 279.93 MW
MAE: 214.67 MW


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ==========================================
# FEATURE SPACE
# ==========================================
# continuous cyclical features
FEATURES = [
    'tod_sin', 'tod_cos', 'dow_sin', 'dow_cos', 'doy_sin', 'doy_cos', 
    'Total_Wind', 'Solar', 
    'rhum', 'DT1', 'DT2', 'DT3', 
    'TL2W_w', 'dTL2W_w_nw', 'HDD', 
    'load_lag_48h', 'load_lag_168h', 'is_school_break', 'is_prev_week_school_break'
]
TARGET = 'load'

# timestamp is the index
if 'timestamp' in df_full.columns:
    df_full = df_full.set_index('timestamp')

df_ridge = df_full.dropna(subset=FEATURES + [TARGET]).copy()

# ==========================================
# Ridge
# ==========================================

ridge_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=10, random_state=42))
])

# ==========================================
# TS-CV
# ==========================================
end_date = pd.to_datetime('2026-03-01 00:00:00', utc=True)
test_start_dates = [end_date - pd.Timedelta(days=i) for i in range(56, 0, -1)]

ridge_forecasts = []
fold_mapes = []

print("Initiating 56-Fold Walk-Forward Ridge Regression Evaluation...")
start_time = time.time()

for i, test_start in enumerate(test_start_dates, 1):
    test_end = test_start + pd.Timedelta(hours=35)
    
    # SPLIT
    train_mask = df_ridge.index < test_start
    test_mask = (df_ridge.index >= test_start) & (df_ridge.index <= test_end)
    
    X_train, y_train = df_ridge.loc[train_mask, FEATURES], df_ridge.loc[train_mask, TARGET]
    X_test, y_test = df_ridge.loc[test_mask, FEATURES], df_ridge.loc[test_mask, TARGET]
    
    # TRAIN & SCALE
    ridge_pipeline.fit(X_train, y_train)
    
    # PREDICT
    preds = ridge_pipeline.predict(X_test)
    
    # STORE RESULTS
    df_pred = pd.DataFrame({
        'timestamp': X_test.index,
        'Actual': y_test.values,
        'Ridge_Prediction': preds,
        'Fold': i
    })
    ridge_forecasts.append(df_pred)
    
    fold_mape = mean_absolute_percentage_error(y_test, preds)
    fold_mapes.append(fold_mape)
    
    if i % 10 == 0 or i == 56:
        print(f"Fold {i}/56 completed. Horizon: {test_start.strftime('%Y-%m-%d')} | MAPE: {fold_mape*100:.2f}%")

# ==========================================
# FINAL RESULTS
# ==========================================
df_ridge_results = pd.concat(ridge_forecasts, ignore_index=True)
global_ridge_mape = mean_absolute_percentage_error(df_ridge_results['Actual'], df_ridge_results['Ridge_Prediction'])
total_time = (time.time() - start_time) / 60

print(f"\n==========================================")
print(f"RIDGE REGRESSION BASELINE RESULTS")
print(f"Total 36h Horizons Tested: 56")
print(f"Execution Time: {total_time:.2f} minutes")
print(f"Global Aggregated MAPE: {global_ridge_mape * 100:.2f}%")
print(f"==========================================")

# Export results
file_name = 'ridge_global_results.csv'
df_ridge_results.to_csv(file_name, index=False)
print(f"Results successfully saved to '{file_name}'")

Initiating 56-Fold Walk-Forward Ridge Regression Evaluation...
Fold 10/56 completed. Horizon: 2026-01-13 | MAPE: 3.87%
Fold 20/56 completed. Horizon: 2026-01-23 | MAPE: 3.93%
Fold 30/56 completed. Horizon: 2026-02-02 | MAPE: 2.42%
Fold 40/56 completed. Horizon: 2026-02-12 | MAPE: 1.69%
Fold 50/56 completed. Horizon: 2026-02-22 | MAPE: 3.00%
Fold 56/56 completed. Horizon: 2026-02-28 | MAPE: 4.31%

RIDGE REGRESSION BASELINE RESULTS
Total 36h Horizons Tested: 56
Execution Time: 0.02 minutes
Global Aggregated MAPE: 5.55%
Results successfully saved to 'ridge_global_results.csv'


In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_percentage_error
import time

# ==========================================
# FEATURE SPACE
# ==========================================
FEATURES = [
    'tod_sin', 'tod_cos', 'dow_sin', 'dow_cos', 'doy_sin', 'doy_cos', 
    'Total_Wind', 'Solar', 't2m', 
    'rhum', 'tcc', 'sp', 'ws', 'swr', 'DT1', 'DT2', 'DT3', 
    'TL2W_w', 'dTL2W_w_nw', 'HDD', 'HDD_mean_D-2', 'HDD_mean_D-7', 
    'load_lag_48h', 'load_lag_168h', 'is_school_break', 'is_prev_week_school_break'
]
TARGET = 'load'

if 'timestamp' in df_full.columns:
    df_full = df_full.set_index('timestamp')

df_elastic = df_full.dropna(subset=FEATURES + [TARGET]).copy()

# ==========================================
# ELASTICNET
# ==========================================
elastic_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', ElasticNet(alpha=1.0, l1_ratio=0.5, random_state=42, max_iter=5000))
])

# ==========================================
# TS-CV
# ==========================================
end_date = pd.to_datetime('2026-03-01 00:00:00', utc=True)
test_start_dates = [end_date - pd.Timedelta(days=i) for i in range(56, 0, -1)]

elastic_forecasts = []
fold_mapes = []
final_coefficients = None

print("Initiating 56-Fold Walk-Forward ElasticNet Evaluation...")
start_time = time.time()

for i, test_start in enumerate(test_start_dates, 1):
    test_end = test_start + pd.Timedelta(hours=35)
    
    train_mask = df_elastic.index < test_start
    test_mask = (df_elastic.index >= test_start) & (df_elastic.index <= test_end)
    
    X_train, y_train = df_elastic.loc[train_mask, FEATURES], df_elastic.loc[train_mask, TARGET]
    X_test, y_test = df_elastic.loc[test_mask, FEATURES], df_elastic.loc[test_mask, TARGET]
    
    elastic_pipeline.fit(X_train, y_train)
    preds = elastic_pipeline.predict(X_test)
    
    # Capture the coefficients from the final fold for feature sparsity analysis
    if i == 56:
        final_coefficients = elastic_pipeline.named_steps['model'].coef_
    
    df_pred = pd.DataFrame({
        'timestamp': X_test.index,
        'Actual': y_test.values,
        'Elastic_Prediction': preds,
        'Fold': i
    })
    elastic_forecasts.append(df_pred)
    
    fold_mape = mean_absolute_percentage_error(y_test, preds)
    fold_mapes.append(fold_mape)
    
    if i % 10 == 0 or i == 56:
        print(f"Fold {i}/56 completed. Horizon: {test_start.strftime('%Y-%m-%d')} | MAPE: {fold_mape*100:.2f}%")

# ==========================================
# FINAL RESULT
# ==========================================
df_elastic_results = pd.concat(elastic_forecasts, ignore_index=True)
global_elastic_mape = mean_absolute_percentage_error(df_elastic_results['Actual'], df_elastic_results['Elastic_Prediction'])
total_time = (time.time() - start_time) / 60

print(f"\n==========================================")
print(f"ELASTICNET BASELINE RESULTS")
print(f"Execution Time: {total_time:.2f} minutes")
print(f"Global Aggregated MAPE: {global_elastic_mape * 100:.2f}%")
print(f"==========================================")

# Export
print("\n--- ELASTICNET FEATURE SELECTION (Fold 56) ---")
coef_df = pd.DataFrame({'Feature': FEATURES, 'Weight': final_coefficients})
coef_df['Absolute_Weight'] = coef_df['Weight'].abs()
coef_df = coef_df.sort_values(by='Absolute_Weight', ascending=False)
print(coef_df[['Feature', 'Weight']].to_string(index=False))

df_elastic_results.to_csv('elasticnet_global_results.csv', index=False)

Initiating 56-Fold Walk-Forward ElasticNet Evaluation...
Fold 10/56 completed. Horizon: 2026-01-13 | MAPE: 6.56%
Fold 20/56 completed. Horizon: 2026-01-23 | MAPE: 5.05%
Fold 30/56 completed. Horizon: 2026-02-02 | MAPE: 7.24%
Fold 40/56 completed. Horizon: 2026-02-12 | MAPE: 3.59%
Fold 50/56 completed. Horizon: 2026-02-22 | MAPE: 3.01%
Fold 56/56 completed. Horizon: 2026-02-28 | MAPE: 4.68%

ELASTICNET BASELINE RESULTS
Execution Time: 0.08 minutes
Global Aggregated MAPE: 6.66%

--- ELASTICNET FEATURE SELECTION (Fold 56) ---
                  Feature     Weight
                   TL2W_w  84.257252
            load_lag_168h  79.944746
             load_lag_48h  75.605035
                      DT1 -66.406952
                  tod_cos -55.163994
                  dow_sin  47.258770
             HDD_mean_D-2  37.758139
                       ws  35.526024
                  tod_sin -35.395129
               Total_Wind  34.612553
                      HDD  31.761674
                  doy_cos  

In [11]:
coef_df

,Feature,Weight,Absolute_Weight
19,TL2W_w,84.783893,84.783893
25,load_lag_168h,80.306558,80.306558
24,load_lag_48h,75.884900,75.884900
16,DT1,-64.736925,64.736925
1,tod_cos,-54.751104,54.751104
2,dow_sin,49.520686,49.520686
22,HDD_mean_D-2,38.125214,38.125214
0,tod_sin,-35.907635,35.907635
5,doy_cos,32.574931,32.574931
21,HDD,31.266313,31.266313
